In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM
from typing import Optional, Iterable, Dict, List
import os

import pandas as pd
import numpy as np
import warnings
import re
import matplotlib.pyplot as plt
import seaborn as sns
from cmcrameri import cm
import massbalancemachine as mbm
import logging
import torch.nn as nn
from skorch.helper import SliceDataset
from datetime import datetime
from skorch.callbacks import EarlyStopping, LRScheduler, Checkpoint
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset
import pickle 
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
from matplotlib.lines import Line2D

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.device_count()==0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"WITHIN_REGION"
print(f"Base directory for data: {BASE_DIR}")

# make directories if they don't exist
os.makedirs(BASE_DIR, exist_ok=True)

## Within region modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"].head()

In [ ]:
df_all = pd.concat(dfs.values(), ignore_index=True)

df_unique = (df_all[["RGIId",
                     "RGI_REGION"]].drop_duplicates().reset_index(drop=True))

out_path = os.path.join(cfg.dataPath, "WGMS/RGIids_stakes_all_regions.csv")
df_unique.to_csv(out_path, index=False)
print(f"Saved {len(df_unique)} unique RGI IDs to {out_path}")
df_unique.head()

In [ ]:
# run it
summarize_and_plot_all_regions(dfs)

In [ ]:
# run it
plot_mb_distributions_all_regions(dfs)

### Separate into train/test glaciers:
Separate using topoclimatic distance.

In [ ]:
dfs_grouped = {
    "NOR": dfs["08"],
    "SJM": dfs["07"],
    "ISL": dfs["06"],
    "CEU": dfs["11"],
    "ALA": dfs["01"],
    "CENTRALASIA": dfs["13"],
}

MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc"),
}
# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

# Transform data to monthly format (run or load data):
paths_HMA = {
    "era5_climate_data":
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
}
# Check that all these files exists
for key, path in paths_HMA.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

MONTHLY_COLS = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
    "ELEVATION_DIFFERENCE",
]
STATIC_COLS = ["aspect", "slope", "svf"]
FEATURE_COLS = MONTHLY_COLS + STATIC_COLS

In [ ]:
paths_multi = {
    "EU_US_CANADA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_EU_US_CANADA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_EU_US_CANADA.nc"),
    },
    "HMA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
    },
}

In [ ]:
# A bit "dirty coding" but I want the monthly dataframes for each whole region
# so it's easier to run like this:
PAIRS = [("ISL", "NOR"), ("NOR", "ISL"), ("ISL", "CH"), ("ISL", "ALA"),
         ("ISL", "CENTRALASIA"), ("ISL", "SJM")]
PAIR_KEYS = [
    "ISL_to_region", "NOR_to_ISL", "ISL_to_CH", "ISL_to_ALA",
    "ISL_to_CENTRALASIA", "ISL_to_SJM"
]

print(f"Pairs: (N = {len(PAIRS)})", PAIRS)
print("Pair keys:", PAIR_KEYS)

# load all stake dfs once
dfs = {
    rid: load_stakes_for_rgi_region(cfg, rid)
    for rid in tqdm(RGI_REGIONS.keys(), desc="Loading stake regions")
}

In [ ]:
# Step 1: collect all unique codes across all pairs
WITHIN_CODES = ["ALA", "CENTRALASIA", "CH", "ISL", "NOR", "SJM"]

# Step 2: compute monthly data once per code
run_flag_by_code = {code: False for code in WITHIN_CODES}

monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    region_codes=WITHIN_CODES,
    run_flag_by_code=run_flag_by_code,
)

In [ ]:
def split_pool_holdout_by_id(
    df_region: pd.DataFrame,
    id_col: str = "ID",
    holdout_frac: float = 0.2,
    seed: int = 0,
) -> dict:
    """
    Random holdout split at the individual stake (ID) level.
    Intended as a fallback for regions with too few glaciers for a
    meaningful glacier-level split (e.g. SJM with 3 glaciers).
    """
    all_ids = sorted(df_region[id_col].unique())
    n_total_meas = len(df_region)
    target_holdout = int(np.round(holdout_frac * n_total_meas))

    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(all_ids)

    holdout_ids, pool_ids = [], []
    holdout_meas_count = 0

    for id_ in shuffled:
        id_meas = (df_region[id_col] == id_).sum()
        if holdout_meas_count < target_holdout:
            holdout_ids.append(id_)
            holdout_meas_count += id_meas
        else:
            pool_ids.append(id_)

    actual_frac = holdout_meas_count / n_total_meas

    print(f"  Total measurements   : {n_total_meas}")
    print(f"  Target holdout meas  : {target_holdout} ({holdout_frac:.0%})")
    print(
        f"  Holdout : {len(holdout_ids)} IDs, {holdout_meas_count} measurements ({actual_frac:.1%})"
    )
    print(
        f"  Pool    : {len(pool_ids)} IDs, {n_total_meas - holdout_meas_count} measurements ({1-actual_frac:.1%})"
    )

    return {
        "holdout_glaciers": holdout_ids,
        "pool_glaciers": pool_ids,
        "n_meas_holdout": int(holdout_meas_count),
        "n_meas_pool": int(n_total_meas - holdout_meas_count),
        "actual_holdout_frac": float(actual_frac),
        "sinkhorn_holdout_vs_region": float("nan"),
        "sinkhorn_pool_vs_region": float("nan"),
    }


def prepare_within_region_pairs_from_cache(
    cfg,
    monthly_cache,
    within_codes,
    holdout_frac=0.20,
    n_seeds=100,
    min_frac=0.15,
    max_frac=0.35,
    glacier_count_threshold=5,
):
    # Build scalers and blur from the monthly_cache data,
    # mirroring the cross-region notebook pattern
    # Reformat monthly_cache into the structure expected by build_global_scalers_multi_source_simple
    res_for_scalers = {
        code: {
            "data_monthly": monthly_cache[code]["data_monthly"]
        }
        for code in monthly_cache
    }

    scaler_m, scaler_s, _ = build_global_scalers_multi_source_simple(
        res_for_scalers,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
    )
    _, _, blur_joint = estimate_global_bandwidths_simple(
        res_for_scalers,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        seed=cfg.seed,
    )
    print(f"Estimated blur_joint={blur_joint:.4f}")

    res_by_code = {}
    split_info_by_code = {}

    for code in within_codes:
        code = code.upper()

        print(f"\n{'='*50}")
        print(f"Within-region: {code}")

        if code not in monthly_cache:
            raise KeyError(f"Code not in monthly_cache: {code}")

        data_monthly = monthly_cache[code]["data_monthly"].copy()
        data_monthly_aug = monthly_cache[code]["data_monthly_aug"].copy()
        head_pad = monthly_cache[code]["months_head_pad"]
        tail_pad = monthly_cache[code]["months_tail_pad"]

        n_glaciers = data_monthly["GLACIER"].nunique()
        use_id_split = n_glaciers < glacier_count_threshold
        split_unit = "ID" if use_id_split else "GLACIER"

        if use_id_split:
            print(
                f"  ⚠ Only {n_glaciers} glaciers — falling back to ID-level split"
            )

        if use_id_split:
            best_split = split_pool_holdout_by_id(
                df_region=data_monthly,
                id_col="ID",
                holdout_frac=holdout_frac,
                seed=cfg.seed,
            )
            best_score = float("nan")
        else:
            best_split, best_score = None, np.inf
            for s in range(n_seeds):
                split = split_pool_holdout_sinkhorn(
                    df_region=data_monthly,
                    monthly_cols=MONTHLY_COLS,
                    static_cols=STATIC_COLS,
                    scaler_m=scaler_m,
                    scaler_s=scaler_s,
                    blur_joint=blur_joint,
                    pool_frac=
                    holdout_frac,  # note: renamed from holdout_frac → pool_frac
                    n_restarts=50,
                    seed=s,
                )
                actual_frac = split["actual_holdout_frac"]
                if not (min_frac <= actual_frac <= max_frac):
                    continue
                score = (split["sinkhorn_holdout_vs_region"] +
                         split["sinkhorn_pool_vs_region"])
                if score < best_score:
                    best_score = score
                    best_split = split

            if best_split is None:
                raise RuntimeError(
                    f"No valid split found for {code} in {n_seeds} seeds "
                    f"(holdout_frac target={holdout_frac}, window=[{min_frac}, {max_frac}])"
                )

        test_units = sorted(best_split["holdout_glaciers"])
        train_units = sorted(best_split["pool_glaciers"])

        print(f"  Split on          : {split_unit}")
        print(f"  Train units       : {len(train_units)}")
        print(f"  Test  units       : {len(test_units)}")
        print(f"  Holdout frac      : {best_split['actual_holdout_frac']:.1%}")
        print(
            f"  Sinkhorn(holdout) : {best_split['sinkhorn_holdout_vs_region']:.4f}"
        )
        print(
            f"  Sinkhorn(pool)    : {best_split['sinkhorn_pool_vs_region']:.4f}"
        )

        def _make_split(data, test_units, split_on):
            dataloader = mbm.dataloader.DataLoader(
                cfg,
                data=data,
                random_seed=cfg.seed,
                meta_data_columns=cfg.metaData)
            _, test_set, train_set = get_CV_splits(
                dataloader,
                test_split_on=split_on,
                test_splits=test_units,
                random_state=cfg.seed,
            )
            df_tr = train_set["df_X"].copy()
            df_tr["y"] = train_set["y"]
            df_te = test_set["df_X"].copy()
            df_te["y"] = test_set["y"]
            return df_tr, df_te

        df_train, df_test = _make_split(data_monthly, test_units, split_unit)
        df_train_aug, df_test_aug = _make_split(data_monthly_aug, test_units,
                                                split_unit)

        res_by_code[code] = {
            "data_monthly": data_monthly,
            "df_train": df_train,
            "df_test": df_test,
            "data_monthly_aug": data_monthly_aug,
            "df_train_aug": df_train_aug,
            "df_test_aug": df_test_aug,
            "train_glaciers": train_units,
            "test_glaciers": test_units,
            "split_on": split_unit,
            "months_head_pad": head_pad,
            "months_tail_pad": tail_pad,
        }
        split_info_by_code[code] = {
            "train_glaciers": train_units,
            "test_glaciers": test_units,
            "split_on": split_unit,
            "best_score": best_score,
            "actual_holdout_frac": best_split["actual_holdout_frac"],
            "sinkhorn_holdout": best_split["sinkhorn_holdout_vs_region"],
            "sinkhorn_pool": best_split["sinkhorn_pool_vs_region"],
        }

        print(f"  Train rows        : {len(df_train)}")
        print(f"  Test  rows        : {len(df_test)}")

    return res_by_code, split_info_by_code

In [ ]:
RECOMPUTE_SPLITS = False
SPLITS_CACHE = BASE_DIR / "within_region_splits_cache.pkl"

if RECOMPUTE_SPLITS or not SPLITS_CACHE.exists():
    res_within_by_code, split_info_by_code = prepare_within_region_pairs_from_cache(
        cfg=cfg,
        monthly_cache=monthly_cache,
        within_codes=WITHIN_CODES,
    )
    with open(SPLITS_CACHE, "wb") as f:
        pickle.dump(
            {
                "res_within_by_code": res_within_by_code,
                "split_info_by_code": split_info_by_code,
            }, f)
    print(f"Saved splits to cache: {SPLITS_CACHE}")
else:
    with open(SPLITS_CACHE, "rb") as f:
        cache = pickle.load(f)
    res_within_by_code = cache["res_within_by_code"]
    split_info_by_code = cache["split_info_by_code"]
    print(f"Loaded splits from cache: {SPLITS_CACHE}")

In [ ]:
# 3. extract split info
TEST_GLACIERS_BY_CODE = {
    code: res["test_glaciers"]
    for code, res in res_within_by_code.items()
}
gl_splits = {
    code: [res["test_glaciers"], res["train_glaciers"]]
    for code, res in res_within_by_code.items()
}

# 4. assemble res_all
res_all = {
    code: {
        "data_monthly": monthly_cache[code]["data_monthly"],
        "data_monthly_aug": monthly_cache[code]["data_monthly_aug"],
        "df_train": res["df_train"],
        "df_test": res["df_test"],
        "df_train_aug": res["df_train_aug"],
        "df_test_aug": res["df_test_aug"],
        "train_glaciers": res["train_glaciers"],
        "test_glaciers": res["test_glaciers"],
        "months_head_pad": monthly_cache[code]["months_head_pad"],
        "months_tail_pad": monthly_cache[code]["months_tail_pad"],
    }
    for code, res in res_within_by_code.items()
}

#### Feature overlap:

In [ ]:
# Example usage:
# res_all is what you got from prepare_monthlies_for_all_regions(...)
figs = plot_overlap_for_all_results(
    results_dict=res_all,
    cfg=cfg,
    STATIC_COLS=STATIC_COLS,
    MONTHLY_COLS=MONTHLY_COLS,
    n_iter=1000,
)

In [ ]:
figs_kde = plot_feature_overlap_all_regions(res_all, STATIC_COLS, MONTHLY_COLS)

## LSTM model
### LSTM datasets:

In [ ]:
logs_cache_dir = BASE_DIR / "LSTM_cache"
logs_cache_dir.mkdir(parents=True, exist_ok=True)

lstm_assets = build_or_load_lstm_all(
    cfg=cfg,
    res_all=res_all,
    MONTHLY_COLS=MONTHLY_COLS,
    STATIC_COLS=STATIC_COLS,
    cache_dir=logs_cache_dir,
    force_recompute=False,
    include_atomic=True,
    include_groups=False,
)

### LSTM parameters:

In [ ]:
default_params = {
    'Fm': 8,
    'Fs': 3,
    'hidden_size': 128,
    'num_layers': 1,
    'bidirectional': False,
    'dropout': 0.0,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'lr': 0.001,
    'weight_decay': 1e-05,
    'loss_name': 'neutral',
    'two_heads': False,
    'head_dropout': 0.1,
    'loss_spec': None
}

GS_RESULTS_DIR = Path(cfg.dataPath) / path_cache / "AdapterLSTM_7/GS_results"

log_path_gs_results = {
    "ISL":
    GS_RESULTS_DIR / 'lstm_param_search_progress_OOS_ISL_2026-02-11.csv',
    "NOR":
    GS_RESULTS_DIR / 'lstm_param_search_progress_OOS_NOR_2026-02-09.csv',
    "CH": GS_RESULTS_DIR / 'lstm_param_search_progress_region_2026-02-18.csv',
}

params_by_key = {}
for code in WITHIN_CODES:
    params = copy.deepcopy(default_params)
    log_path = log_path_gs_results.get(code)
    if log_path and log_path.exists():
        print(f"Loading tuned params for {code}")
        best_params = get_best_params_for_lstm(log_path,
                                               select_by="avg_test_loss")
        params.update(best_params)
    else:
        print(f"No grid-search log for {code} — using defaults")
    params_by_key[code] = params

print(sorted(params_by_key.keys()))

### Train model:

In [ ]:
MODEL_DATE = "2026-05-21"

# LSTM
models_lstm, infos_lstm = train_within_region_models_all(
    cfg=cfg,
    lstm_assets_by_key=lstm_assets,
    params_by_key=params_by_key,
    device=device,
    epochs=150,
    force_retrain=False,
    models_dir=BASE_DIR / "models",
    prefix="lstm_within",
    date=MODEL_DATE,
    model_type="lstm",
)

# Transformer
best_params_gs = {
    'Fm': 8,
    'Fs': 3,
    'd_model': 128,
    'nhead': 8,
    'num_layers': 3,
    'dim_feedforward': 128,
    'dropout': 0.1,
    'static_layers': 2,
    'static_hidden': 64,
    'static_dropout': 0.0,
    'head_dropout': 0.1,
    'lr': 0.0005,
    'weight_decay': 1e-05,
    'two_heads': False,
    'loss_spec': None,
    'T_max': 32
}
transformer_params_by_key = {code: best_params_gs for code in WITHIN_CODES}

models_tf, infos_tf = train_within_region_models_all(
    cfg=cfg,
    lstm_assets_by_key=lstm_assets,
    params_by_key=transformer_params_by_key,
    device=device,
    epochs=150,
    force_retrain=False,
    models_dir=BASE_DIR / "models",
    prefix="transformer_within",
    date=MODEL_DATE,
    model_type="transformer",
)

### Evaluate on test:

In [ ]:
df_metrics, preds_by_key, figs_by_key, fig_grid = evaluate_all_models(
    cfg=cfg,
    models_by_key=models_tf,
    lstm_assets_by_key=lstm_assets,
    device=device,
    save_dir="figures/eval_within_region/transformer",
    ax_xlim=(-16, 10),
    ax_ylim=(-16, 10),
    grid_shape=(2, 3),
    grid_figsize=(25, 12),
    complement_key="",
)

display(df_metrics)

In [ ]:
df_metrics_lstm, preds_by_key_lstm, figs_by_key_lstm, fig_grid_lstm = evaluate_all_models(
    cfg=cfg,
    models_by_key=models_lstm,
    lstm_assets_by_key=lstm_assets,
    device=device,
    save_dir="figures/eval_within_region/lstm",
    ax_xlim=(-16, 10),
    ax_ylim=(-16, 10),
    grid_shape=(2, 3),
    grid_figsize=(25, 12),
    complement_key="",
)

display(df_metrics_lstm)